# HW2.3: Analyzing Data Directly from S3

Reads the hourly `.parquet` files directly from your own S3 bucket (never from the local `data/` folder) and produces two plots of hourly event counts.

**Requires the `s3fs` package** for `pd.read_parquet()` to be able to read `s3://` URIs. On your EC2 instance, run `uv add s3fs` before executing this notebook.

In [ ]:
import boto3
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [ ]:
netid = "kt917"
bucket_name = f"dsan6000-{netid}"

s3 = boto3.client("s3", region_name="us-east-1")

In [ ]:
# List the hourly files in your bucket
response = s3.list_objects_v2(Bucket=bucket_name, Prefix="wikipedia-hourly/")
keys = [obj["Key"] for obj in response["Contents"]]
print(f"Found {len(keys)} files in your bucket")

In [ ]:
# Read each hourly file directly from S3 (NOT from the local data/ folder) and concatenate
dfs = []
for key in keys:
    uri = f"s3://{bucket_name}/{key}"
    dfs.append(pd.read_parquet(uri))

all_events = pd.concat(dfs, ignore_index=True)
print(all_events.shape)
all_events.head()

In [ ]:
# IMPORTANT: check the printed column names above and adjust these two variables
# to match your actual dataframe (these are common guesses, not confirmed for your dataset)
print(all_events.columns.tolist())

timestamp_col = "timestamp"  # e.g. could actually be 'meta.dt', 'dt', 'event_timestamp', etc.
type_col = "type"            # e.g. could actually be 'event_type' or similar

all_events["hour"] = pd.to_datetime(all_events[timestamp_col]).dt.floor("h")

os.makedirs("images", exist_ok=True)

In [ ]:
# Plot 1: total events per hour
hourly_totals = all_events.groupby("hour").size().reset_index(name="count")

plt.figure(figsize=(12, 6))
sns.lineplot(data=hourly_totals, x="hour", y="count")
plt.title("Total Wikipedia Events per Hour")
plt.xlabel("Hour")
plt.ylabel("Event Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("images/hourly-events.svg")
plt.savefig("images/hourly-events.png")
plt.show()

In [ ]:
# Plot 2: hourly events by type
hourly_by_type = all_events.groupby(["hour", type_col]).size().reset_index(name="count")

plt.figure(figsize=(12, 6))
sns.lineplot(data=hourly_by_type, x="hour", y="count", hue=type_col)
plt.title("Wikipedia Events per Hour by Type")
plt.xlabel("Hour")
plt.ylabel("Event Count")
plt.xticks(rotation=45)
plt.legend(title="Event Type")
plt.tight_layout()
plt.savefig("images/events-by-type.svg")
plt.savefig("images/events-by-type.png")
plt.show()